In [1]:
%cd ../src
%load_ext autoreload
%autoreload 2

/Users/martinbierey/git/dax-hoc/src


### 1) Preparation 

In [3]:
import logging
import sys
import uuid
from datetime import datetime, timezone
from pathlib import Path
from sqlalchemy import create_engine, select
from db import bootstrap, get_engine, get_session, read_all 
from db.models import Company, EarningsEvent, NewsItem



from llm.registry import call_llm
from db.models import Company, EarningsEvent, ExpectationsSnapshot
import json

from expectations.researcher import research,_PROMPT,call_llm,_parse_sections
from db.models import Company, EarningsEvent, ExpectationsSnapshot
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s — %(message)s")
logger = logging.getLogger(__name__)

NEWS_ITEM_ID = "e1ec472f-4aed-4662-a5d3-c9cf3f9d5c67_en"

#### 2) Preparation 

In [8]:
engine = get_engine()

In [9]:
company = Company(
    isin="DE000A2LQ884",
    name="AUTO1 Group SE",
    index_name="MDAX",
    industry="Automotive E-Commerce",
    description="Europe's leading digital automotive marketplace for used cars.",
    synced_at=datetime.now(timezone.utc),
)

event = EarningsEvent(
    id=str(uuid.uuid4()),
    isin="DE000A2LQ884",
    fiscal_period="2026-Q1",
    event_type="quarterly",
    actual_release_at=datetime(2026, 2, 25, 7, 30, tzinfo=timezone.utc),
    status="released",
)

In [10]:
prompt = _PROMPT.format(
        company_name=company.name,
        isin=company.isin,
        index_name=company.index_name,
        industry=company.industry or "N/A",
        fiscal_period=event.fiscal_period,
        event_type=event.event_type,
        expected_date=str(event.expected_date),
    )

In [11]:
content, run_id = call_llm(
        role="expectations_researcher",
        prompt=prompt,
        engine=engine,
        earnings_event_id=event.id,
        web_search=True,
    )

INFO llm.registry — LLM call: role=expectations_researcher provider=openai model=gpt-5.4 web_search=True
INFO httpx — HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO llm.registry — LLM done: role=expectations_researcher tokens_in=249 tokens_out=717 latency=17525ms


In [12]:
narrative_md, trade_thesis_md, sources = _parse_sections(content)
snapshot = ExpectationsSnapshot(
        id=str(uuid.uuid4()),
        earnings_event_id=event.id,
        gathered_at=datetime.now(timezone.utc),
        narrative_md=narrative_md,
        trade_thesis_md=trade_thesis_md,
        sources_json=json.dumps(sources),
        raw_research_text=content,
        llm_run_id=run_id,
    )

In [13]:
narrative_md

'AUTO1 Group SE, a leading European digital automotive platform, has demonstrated consistent growth across key performance indicators in recent quarters. In Q4 2025, the company sold 219,110 units, marking a 19.7% year-over-year increase, and achieved a gross profit of €265.4 million, up 31.8% from the previous year. Adjusted EBITDA for the same period was €45.2 million. ([webdisclosure.com](https://www.webdisclosure.com/article/auto1-group-se-etr-auto1-group-reports-record-2025-financial-performance-b7dMOwW6UA7?utm_source=openai))\n\nFor the full year 2025, AUTO1 reported 842,271 units sold, a 22.1% increase year-over-year, with a gross profit of €990.6 million, up 36.7%. The company\'s market share in the European used car market grew to 3.1%. ([webdisclosure.com](https://www.webdisclosure.com/article/auto1-group-se-etr-auto1-group-reports-record-2025-financial-performance-b7dMOwW6UA7?utm_source=openai))\n\nLooking ahead, AUTO1 has provided guidance for 2026, anticipating unit sales 

In [14]:
trade_thesis_md

"Consider purchasing AUTO1 Group SE stock post-earnings announcement if the following conditions are met:\n\n1. **Units Sold Growth**: Q1 2026 units sold exceed 210,000, indicating continued strong demand and market expansion.\n\n2. **Gross Profit Margin Improvement**: Gross profit margin shows a year-over-year increase, reflecting enhanced operational efficiency and pricing strategies.\n\n3. **Adjusted EBITDA Progress**: Adjusted EBITDA demonstrates positive momentum towards the company's profitability targets, aligning with the 2026 guidance.\n\nConversely, even if headline figures appear positive, exercise caution if there is a significant decline in gross profit margin or if the company revises its 2026 guidance downward, as these factors could indicate underlying challenges."

In [ ]:
session = get_session(engine)

try:
    session.add(snapshot)
    session.commit()
finally:
    session.close()

In [ ]:
#### Delete if necessary 
from db import delete_rows
cid = '85455a14-7165-4142-a3e8-9cb85d8fd160'
#print(delete_rows(table='expectations_snapshots',ids=[cid]))